# 03 — HDFS features, split and baselines

- Chronological split: 60% train / 20% validation / 20% test (by block start time)
- Event-count vectors, vocabulary learned from training normal blocks only (+ "unseen" column)
- Recall ceiling: anomalies whose count vector is identical to a normal training vector
- Simple baselines that ML models must beat

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Project root not found")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
OUT = ROOT / "data" / "processed"

from backend.detection.evaluation.metrics import binary_metrics
from backend.detection.evaluation.splits import chronological_split
from backend.detection.features.vectorizer import EventCountVectorizer, parse_events

sequences = pd.read_csv(OUT / "hdfs_sequences.csv")
assert sequences["first_ts"].is_monotonic_increasing, "sequences must be in time order"

events = [parse_events(s) for s in sequences["events"]]
y = (sequences["Label"] == "Anomaly").to_numpy()
lengths = sequences["length"].to_numpy()
print(f"{len(sequences):,} blocks, {y.sum():,} anomalies")

575,061 blocks, 16,838 anomalies


In [2]:
split = chronological_split(len(sequences), train=0.6, validation=0.2)

rows = []
for name, idx in split.items():
    ts = sequences["first_ts"].iloc[idx]
    rows.append({
        "split": name,
        "blocks": len(idx),
        "anomalies": int(y[idx].sum()),
        "anomaly_%": round(100 * y[idx].mean(), 2),
        "from": str(ts.iloc[0]),
        "to": str(ts.iloc[-1]),
    })
display(pd.DataFrame(rows))

train_normal = split["train"][~y[split["train"]]]
print(f"Training set for unsupervised models: {len(train_normal):,} normal blocks "
      f"({int(y[split['train']].sum()):,} training anomalies set aside)")

,split,blocks,anomalies,anomaly_%,from,to
0,train,345036,11158,3.23,81109203518,81111041726
1,validation,115012,4001,3.48,81111041726,81111074233
2,test,115013,1679,1.46,81111074233,81111110413


Training set for unsupervised models: 333,878 normal blocks (11,158 training anomalies set aside)


In [3]:
vectorizer = EventCountVectorizer().fit(events[i] for i in train_normal)
X = vectorizer.transform(events)
print(f"Feature matrix: {X.shape[0]:,} blocks x {X.shape[1]} features")
print(f"Templates in the training vocabulary: {len(vectorizer.vocabulary_)} of 53")

has_unseen = X[:, -1] > 0
rows = []
for name, idx in split.items():
    for label, mask in (("Normal", ~y[idx]), ("Anomaly", y[idx])):
        part = idx[mask]
        rows.append({"split": name, "label": label, "blocks": len(part),
                     "with_unseen_template_%": round(100 * has_unseen[part].mean(), 2) if len(part) else 0})
display(pd.DataFrame(rows))

Feature matrix: 575,061 blocks x 20 features
Templates in the training vocabulary: 19 of 53


,split,label,blocks,with_unseen_template_%
0,train,Normal,333878,0.00
1,train,Anomaly,11158,22.31
2,validation,Normal,111011,0.00
3,validation,Anomaly,4001,15.75
4,test,Normal,113334,0.00
5,test,Anomaly,1679,43.24


In [4]:
row_keys = [row.tobytes() for row in X]
train_normal_keys = {row_keys[i] for i in train_normal}
all_normal_keys = {row_keys[i] for i in np.flatnonzero(~y)}

print(f"Distinct count vectors among training normal blocks: {len(train_normal_keys)}\n")
for name in ("validation", "test"):
    anomalies = split[name][y[split[name]]]
    identical = sum(row_keys[i] in train_normal_keys for i in anomalies)
    print(f"{name:<10} anomalies {len(anomalies):>6,}   identical to a training-normal vector {identical:>5,}   "
          f"-> max recall for any count-based model {100 * (1 - identical / len(anomalies)):.2f}%")

anomalies_all = np.flatnonzero(y)
ambiguous = sum(row_keys[i] in all_normal_keys for i in anomalies_all)
print(f"\nWhole dataset: {ambiguous:,} of {len(anomalies_all):,} anomalies "
      f"({100 * ambiguous / len(anomalies_all):.2f}%) have exactly the same count vector as some normal block")

Distinct count vectors among training normal blocks: 207

validation anomalies  4,001   identical to a training-normal vector     0   -> max recall for any count-based model 100.00%
test       anomalies  1,679   identical to a training-normal vector     0   -> max recall for any count-based model 100.00%

Whole dataset: 319 of 16,838 anomalies (1.89%) have exactly the same count vector as some normal block


In [5]:
min_len, max_len = lengths[train_normal].min(), lengths[train_normal].max()
print(f"Training normal block length range: {min_len}-{max_len} lines\n")

length_rule = (lengths < min_len) | (lengths > max_len)
unseen_rule = has_unseen
lookup_rule = np.array([key not in train_normal_keys for key in row_keys])

baselines = {
    "B1 line count outside training range": length_rule,
    "B2 contains unseen template": unseen_rule,
    "B3 B1 or B2": length_rule | unseen_rule,
    "B4 count vector never seen in training (lookup table)": lookup_rule,
}

rows = []
for name, prediction in baselines.items():
    for part in ("validation", "test"):
        idx = split[part]
        m = binary_metrics(y[idx], prediction[idx])
        rows.append({"baseline": name, "split": part, "precision": round(m.precision, 4),
                     "recall": round(m.recall, 4), "f1": round(m.f1, 4), "fpr": round(m.fpr, 5),
                     "tp": m.tp, "fp": m.fp, "fn": m.fn})
results = pd.DataFrame(rows)
display(results)

Training normal block length range: 13-298 lines



,baseline,split,precision,recall,f1,fpr,tp,fp,fn
0,B1 line count outside training range,validation,1.0000,0.2302,0.3742,0.00000,921,0,3080
1,B1 line count outside training range,test,1.0000,0.6152,0.7618,0.00000,1033,0,646
2,B2 contains unseen template,validation,1.0000,0.1575,0.2721,0.00000,630,0,3371
3,B2 contains unseen template,test,1.0000,0.4324,0.6037,0.00000,726,0,953
4,B3 B1 or B2,validation,1.0000,0.2672,0.4217,0.00000,1069,0,2932
5,B3 B1 or B2,test,1.0000,0.7082,0.8291,0.00000,1189,0,490
6,B4 count vector never seen in training (lookup...,validation,0.9993,1.0000,0.9996,0.00003,4001,3,0
7,B4 count vector never seen in training (lookup...,test,0.9562,1.0000,0.9776,0.00068,1679,77,0


In [6]:
np.savez_compressed(
    OUT / "hdfs_features.npz",
    X=X, y=y, lengths=lengths,
    train=split["train"], validation=split["validation"], test=split["test"],
    train_normal=train_normal,
    feature_names=np.array(vectorizer.feature_names),
    block_ids=sequences["BlockId"].to_numpy().astype(str),
)
results.to_csv(ROOT / "experiments" / "results" / "hdfs_baselines.csv", index=False)
print("Saved hdfs_features.npz and experiments/results/hdfs_baselines.csv")

Saved hdfs_features.npz and experiments/results/hdfs_baselines.csv


## Findings
- Chronological split: train 345,036 (3.23% anomalies), validation 115,012 (3.48%), test 115,013 (1.46%).
  Anomaly rate and type shift over time (B1 recall 0.23 on validation vs 0.62 on test).
- Training normal blocks use only 19 of 53 templates -> 20 features (19 + unseen).
  No normal block in any split contains an unseen template.
- 207 distinct count vectors among 333,878 training normal blocks.
- Recall ceiling 100%: no validation/test anomaly has a count vector identical to a training-normal block
  (whole dataset: 319 anomalies, 1.89%, identical to some normal block).
- Baselines (test F1): B1 line count 0.762, B2 unseen template 0.604, B3 B1|B2 0.829,
  B4 lookup table 0.978 (precision 0.956, recall 1.000).
- HDFS v1 is almost solved by memorising normal count vectors. B4's false positives grow from 3 (validation)
  to 77 (test) as new-but-normal patterns appear: memorisation does not generalise.
- Consequence: ML models on HDFS are judged against B4 and on false positives for new normal patterns;
  full comparison on BGL and the live system, where exact memorisation is impossible.